# 🌐 Folium — Mapas Interactivos con Python
**Curso:** Scientific Computing | Economía Aplicada  
**Tema:** Visualización Geoespacial Dinámica  
**Datasets:** MINING.csv (MINEM) · ENAHO · Solidaridad Health Centers

---
> 📌 **Guía de uso:** Este notebook es la **parte práctica** — la teoría está en el PPT.  
> Cada bloque de código corresponde a un **paso o slide** del PPT.  
> Los `# comentarios` indican el concepto teórico que se está aplicando.


## ⚙️ Paso 0 — Instalación e Importación de Librerías
> 📌 **PPT Slide 3** — Ecosistema completo de librerías

In [ ]:
 #─── Instalar (ejecutar solo la primera vez) ─────────────────────────────────
 !pip install folium
 !pip install branca
 !pip install chardet
 !pip install geopandas


In [ ]:
# ─── Importar librerías ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import chardet                              # PPT Slide 3: detectar encoding

import folium as fm                        # librería principal — mapas interactivos
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap, StripePattern

import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString

import branca as br                        # PPT Slide 9: popups HTML avanzados

print("✅ Librerías cargadas correctamente")
print(f"   folium version: {fm.__version__}")


In [ ]:
from IPython.display import display, HTML
display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))


---
## 📦 Paso 1 — Cargar Datos: Minería del Perú (MINING.csv)
> 📌 **PPT Slide 4** — Primer mapa y tiles | **PPT Slide 3** — Encoding con chardet  
> **Fuente:** MINEM — http://www.minem.gob.pe/_publicaSector.php?idSector=1&idCategoria=24


In [ ]:
# ─── PASO 1A: Detectar encoding (PPT Slide 3 — chardet) ──────────────────────
# IMPORTANTE: Antes de leer cualquier CSV, detectar su encoding

ruta_mining = r"../../../_data/Folium/MINING.csv"

with open(ruta_mining, 'rb') as f:
    rawdata = f.read()
det = chardet.detect(rawdata)
charenc = det['encoding']

print(f"Encoding detectado: {charenc}  |  Confianza: {det['confidence']:.0%}")


In [ ]:
# ─── PASO 1B: Leer el CSV con el encoding correcto ───────────────────────────
MINING = pd.read_csv(ruta_mining, encoding=charenc)

print(f"Dimensiones: {MINING.shape[0]} filas x {MINING.shape[1]} columnas")
print(f"Columnas: {list(MINING.columns)}")
display(MINING.head(5))


In [ ]:
# ─── PASO 1C: Filtrar por zona de interés — Yauli, Junín ─────────────────────
# Técnica: filtrar el dataset para un análisis específico
MINING1 = MINING[MINING.DISTRITO == "YAULI"]

print(f"Empresas mineras en Yauli: {len(MINING1)}")
display(MINING1[['TITULAR','PRODUCTO','LATITUD','LONGITUD']])


---
## 🗺️ Paso 2 — Primer Mapa: fm.Map() y Tipos de Tiles
> 📌 **PPT Slide 4** — fm.Map(): Primer Mapa y Selección de Tiles  
> `location` = coordenadas del centro | `zoom_start` = nivel de zoom (1=mundo, 18=calle) | `control_scale=True` = barra de escala km


In [ ]:
# ─── PASO 2A: Calcular centro del mapa desde los datos ───────────────────────
# PPT Slide 4: centrar el mapa en el promedio de coordenadas del dataset

zoom_start = 12
lat_mining  = MINING1["LATITUD"].mean()    # promedio de latitudes
long_mining = MINING1["LONGITUD"].mean()   # promedio de longitudes

print(f"Centro del mapa → Lat: {lat_mining:.4f} | Lon: {long_mining:.4f}")


In [ ]:
# ─── PASO 2B: Crear el primer mapa (tile: OpenStreetMap) ─────────────────────
# PPT Slide 4: fm.Map() — punto de partida de Folium

a = fm.Map(
    location    = [lat_mining, long_mining],  # ← coordenadas del centro
    tiles       = "OpenStreetMap",            # ← tipo de fondo del mapa
    zoom_start  = zoom_start,                 # ← distancia de vista inicial
    control_scale = True                      # ← barra de escala en km
)

a   # mostrar mapa


In [ ]:
# ─── PASO 2C: Comparar tiles disponibles ──────────────────────────────────────
# PPT Slide 4: tabla de tiles — cada uno tiene un uso recomendado
# "cartodbpositron" → fondo blanco (ideal para choropleth)
# "cartodbdark_matter" → fondo oscuro (ideal para HeatMap)
# "stamenterrain" → relieve (ideal para minería/rural)

tiles_ejemplo = fm.Map(
    location   = [lat_mining, long_mining],
    tiles      = "cartodbpositron",    # ← probar: cambiar el tile aquí
    zoom_start = 8
)

tiles_ejemplo


---
## 📍 Paso 3 — Marcadores: Marker, Circle y Tooltip
> 📌 **PPT Slide 5** — Marcadores: Posicionar Puntos sobre el Mapa  
> **Tooltip** = texto al pasar el cursor | **Popup** = información al hacer click


In [ ]:
# ─── PASO 3A: Marker manual (1 punto con coordenadas fijas) ──────────────────
# PPT Slide 5 — Parte A: Marker básico

tooltip = "Click me!"   # texto que aparece al pasar el cursor

a = fm.Map(location=[lat_mining, long_mining], tiles="OpenStreetMap", zoom_start=12)

# Agregar 2 marcadores manualmente con coordenadas conocidas
fm.Marker(
    [-11.637246, -76.171596],
    popup   = "<i>COMPAÑIA MINERA CASAPALCA S.A.</i>",  # HTML en el popup
    tooltip = tooltip
).add_to(a)

fm.Marker(
    [-11.628891, -76.096077],
    popup   = "<b><i>COMPAÑIA MINERA ARGENTUM S.A.</i></b>",
    tooltip = tooltip
).add_to(a)

a


In [ ]:
# ─── PASO 3B: Marker con loop — iterar todas las filas del DataFrame ─────────
# PPT Slide 5 — Parte B: iterrows() — patrón estándar para múltiples marcadores
# Ícono: fm.Icon(color, icon) — PPT Slide 5 tabla de iconos Font Awesome

a = fm.Map(location=[lat_mining, long_mining], tiles="OpenStreetMap", zoom_start=11)

for index, row in MINING1.iterrows():
    fm.Marker(
        [row['LATITUD'], row['LONGITUD']],
        popup   = row['TITULAR'],                           # nombre de empresa
        icon    = fm.Icon(color="red", icon="info-sign"),   # ícono personalizado
        tooltip = tooltip
    ).add_to(a)

a


In [ ]:
# ─── PASO 3C: Circle — radio en METROS (zona de influencia) ──────────────────
# PPT Slide 5 — Parte C: fm.Circle vs fm.CircleMarker
# fm.Circle → radio en metros reales (cambia con el zoom)
# fm.CircleMarker → radio en píxeles (fijo en pantalla)

a = fm.Map(location=[lat_mining, long_mining], tiles="OpenStreetMap", zoom_start=10)

for index, row in MINING1.iterrows():
    # Marker encima
    fm.Marker(
        [row['LATITUD'], row['LONGITUD']],
        popup = row['TITULAR'],
        icon  = fm.Icon(color="red", icon="info-sign")
    ).add_to(a)
    
    # Círculo de 10km de radio (zona de influencia)
    fm.Circle(
        [row['LATITUD'], row['LONGITUD']],
        popup      = row['TITULAR'],
        radius     = 10000,           # ← 10 km en metros
        fill       = True,
        fill_color = "#3186cc",
        color      = "#3186cc",
        tooltip    = "Zona de influencia 10 km"
    ).add_to(a)

a


---
##  Paso 4 — Choropleth: Mapa de Pobreza por Distritos
>  **PPT Slide 6** — Choropleth: Visualizando Indicadores Económicos por Territorio  
> Requiere: GeoJSON (geometría) + CSV (indicadores) + clave de unión `key_on`


### 4.1 — Cargar GeoJSON (geometría del Perú)
> GeoJSON es el formato que Folium usa para dibujar los polígonos de distritos y departamentos.  
> Fuente: https://visor.geoperu.gob.pe/ | https://ide.inei.gob.pe/


In [ ]:
# ─── PASO 4A: Cargar GeoJSON distrital ───────────────────────────────────────
# PPT Slide 6: geo_data en formato GeoJSON — obligatorio para fm.Choropleth
# Descargar desde: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

ruta_distritos = r"../../../_data/Folium\peru_distrital_simple.geojson"  # ← ajustar ruta

try:
    distritos = gpd.read_file(ruta_distritos)
    distritos1 = distritos[['IDDIST', 'geometry']].rename({'IDDIST': 'UBIGEO1'}, axis=1)
    distritos1['UBIGEO1'] = distritos1['UBIGEO1'].astype(str).astype(np.int64)
    print(f"✅ Distritos cargados: {len(distritos1)}")
    display(distritos1.head(3))
    
except Exception as e:
    print(f"⚠️  Ajustar ruta del GeoJSON: {e}")
    print("   Descargar desde: https://www.geogpsperu.com")


In [ ]:
# ─── PASO 4B: Cargar GeoJSON departamental ───────────────────────────────────
ruta_dpto = r"../../../_data/Folium\peru_departamental_simple.geojson"   # ← ajustar ruta

try:
    dpto = gpd.read_file(ruta_dpto)
    dpto1 = dpto[['FIRST_IDDP', 'geometry']].rename({'FIRST_IDDP': 'UBIGEO2'}, axis=1)
    dpto1['UBIGEO2'] = dpto1['UBIGEO2'] + "0000"
    dpto1['UBIGEO2'] = dpto1['UBIGEO2'].astype(str).astype(np.int64)
    print(f"✅ Departamentos cargados: {len(dpto1)}")
    display(dpto1.head(3))
    
except Exception as e:
    print(f"⚠️  Ajustar ruta del GeoJSON: {e}")


### 4.2 — Cargar datos económicos (Poverty.csv)

In [ ]:
# ─── PASO 4C: Cargar indicadores económicos ──────────────────────────────────
ruta_poverty = r"../../../_data/Folium\Poverty.csv"   # ← ajustar ruta

try:
    with open(ruta_poverty, 'rb') as f:
        det = chardet.detect(f.read())
    poverty = pd.read_csv(ruta_poverty, encoding=det['encoding'])
    print(f"✅ Poverty cargado: {poverty.shape}")
    print(f"   Columnas: {list(poverty.columns)}")
    display(poverty.head(3))
    
except Exception as e:
    print(f"⚠️  Ajustar ruta de Poverty.csv: {e}")


In [ ]:
# ─── PASO 4D: Preparar datos a nivel departamental ───────────────────────────
# Eliminar duplicados, mantener un registro por departamento

poverty2 = poverty.drop_duplicates(subset=['UBIGEO2']).copy()

# Limpiar IDH (puede tener '#N/D')
import pandas as pd, numpy as np
poverty2['IDH'] = pd.to_numeric(poverty2['IDH'].replace('#N/D', np.nan), errors='coerce')
poverty2['PBI_PC'] = pd.to_numeric(poverty2['PBI_PC'].astype(str).replace('#N/D', np.nan), errors='coerce')

print(f"Departamentos: {len(poverty2)}")
display(poverty2[['DEPARTAMENTO','IDH','PBI_PC','POVERTY_RATE']].head(8))


### 4.3 — Choropleth con LayerControl (2 indicadores en 1 mapa)

In [ ]:
# ─── PASO 4F: Choropleth + LayerControl — múltiples capas ────────────────────
# PPT Slide 6 — Paso 4: LayerControl permite alternar entre capas

z = fm.Map(location=[lat_palacio, long_palacio], tiles='cartodbpositron', zoom_start=5)

poverty_idh = poverty.copy()
poverty_idh['IDH'] = pd.to_numeric(poverty_idh['IDH'].replace('#N/D', np.nan), errors='coerce')

# Capa 1: Tasa de Pobreza
fm.Choropleth(
    geo_data     = distritos1,
    data         = poverty,
    columns      = ['UBIGEO1', 'POVERTY_RATE'],
    key_on       = "feature.properties.UBIGEO1",
    fill_color   = "YlOrRd",
    fill_opacity = 0.8,
    line_opacity = 0.2,
    legend_name  = "Tasa de Pobreza (%)",
    name         = "Pobreza",    # ← nombre en el panel de capas
    overlay      = True,
    highlight    = True
).add_to(z)

# Capa 2: IDH (oculta por defecto)
fm.Choropleth(
    geo_data     = distritos1,
    data         = poverty_idh,
    columns      = ['UBIGEO1', 'IDH'],
    key_on       = "feature.properties.UBIGEO1",
    fill_color   = "Blues",
    fill_opacity = 0.8,
    line_opacity = 0.2,
    legend_name  = "Índice de Desarrollo Humano (IDH)",
    name         = "IDH",
    overlay      = True,
    highlight    = True,
    show         = False         # ← oculta por defecto
).add_to(z)

# Panel de control de capas
fm.LayerControl().add_to(z)

z


### 4.4 — Choropleth Departamental con Quantile Bins + Tooltip Interactivo

In [ ]:
# ─── PASO 4G: Calcular bins por quantiles (PPT Slide 6 — Paso 3) ─────────────
# PPT: Usar quantiles es la práctica correcta en análisis económico
# Distribución asimétrica → intervalos iguales ocultarían concentraciones

bins = list(poverty2["IDH"].quantile([0, 0.2, 0.4, 0.6, 0.8, 1]))
print("Bins por quintiles del IDH:")
print([round(b, 3) for b in bins])


In [ ]:
# ─── PASO 4H: Choropleth departamental con bins + Tooltip interactivo ─────────
# PPT Slide 7: Tooltip = información al pasar el cursor (hover)
# Patrón: merge → .to_json() → GeoJson() → GeoJsonTooltip

z = fm.Map(location=[lat_palacio, long_palacio], zoom_start=5)

# Choropleth base (IDH por departamento)
fm.Choropleth(
    geo_data     = dpto1,
    data         = poverty2,
    columns      = ["UBIGEO2", "IDH"],
    key_on       = "feature.properties.UBIGEO2",
    fill_color   = "Reds",
    fill_opacity = 0.8,
    legend_name  = "Índice de Desarrollo Humano (IDH)",
    bins         = bins,            # ← distribución por quintiles
    reset        = True
).add_to(z)

# ─── Tooltip interactivo (PPT Slide 7) ───────────────────────────────────────
# Paso 1: merge geometría + datos → necesario para que el tooltip tenga los datos
data_both = pd.merge(dpto1, poverty2, how="inner", on="UBIGEO2")
data_json = data_both.to_json()    # ← OBLIGATORIO: convertir a JSON para Folium

# Paso 2: estilo normal de cada región (casi transparente)
style_function = lambda x: {
    'fillColor'  : '#ffffff',
    'color'      : '#000000',
    'fillOpacity': 0.1,
    'weight'     : 0.1
}

# Paso 3: estilo al pasar el cursor (hover highlight)
highlight_function = lambda x: {
    'fillColor'  : '#000000',
    'color'      : '#000000',
    'fillOpacity': 0.5,
    'weight'     : 0.1
}

# Paso 4: capa GeoJson con tooltip
details = fm.features.GeoJson(
    data               = data_json,
    style_function     = style_function,
    control            = False,
    highlight_function = highlight_function,
    tooltip            = fm.features.GeoJsonTooltip(
        fields   = ['DEPARTAMENTO', 'IDH', 'POVERTY_RATE'],   # columnas a mostrar
        aliases  = ['Departamento', 'IDH', 'Tasa Pobreza (%)'],
        style    = "background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"
    )
)

# Paso 5: agregar al mapa (keep_in_front → evita que choropleth tape el tooltip)
z.add_child(details)
z.keep_in_front(details)   # ← PPT Slide 7: error frecuente olvidar esto

fm.LayerControl().add_to(z)

z


In [ ]:
# ─── PASO 4I: Guardar el mapa como HTML independiente ─────────────────────────
# El mapa se puede abrir en cualquier navegador sin necesitar Python

z.save("IDH_Pobreza_Peru.html")
print("💾 Guardado: IDH_Pobreza_Peru.html")
print("   Abrir en el navegador para ver el mapa interactivo")


---
##  Paso 5 — HeatMap y MarkerCluster: Hogares Beneficiarios (ENAHO)
>  **PPT Slide 8** — HeatMap y MarkerCluster: Visualizar Densidad de Puntos  
> **Fuente:** INEI — Encuesta Nacional de Hogares (ENAHO)  
> `bono_uni = 1` → hogar recibió el Bono Universal COVID


In [ ]:
# ─── PASO 5A: Cargar ENAHO (PPT Slide 3 — encoding con chardet) ──────────────

ruta_enaho = r"../../../_data/Folium\enaho.csv"

with open(ruta_enaho, 'rb') as f:
    det = chardet.detect(f.read())

ENAHO = pd.read_csv(ruta_enaho, encoding=det['encoding'])

print(f"Dimensiones: {ENAHO.shape}")
print(f"Columnas: {list(ENAHO.columns)}")
display(ENAHO.head(3))


In [ ]:
# ─── PASO 5B: Filtrar beneficiarios del Bono Universal ───────────────────────
# bono_uni == 1 → hogar que recibió el bono
# bono_uni == 0 → hogar que NO recibió el bono

bono_uni = ENAHO[ENAHO.bono_uni == 1]

print(f"Total hogares encuestados: {len(ENAHO)}")
print(f"Beneficiarios Bono Universal (bono_uni=1): {len(bono_uni)}")

display(bono_uni[['latitud','longitud','nbi1','nbi2','nbi3']].head(5))


In [ ]:
# ─── PASO 5C: Convertir coordenadas a lista de tuplas ────────────────────────
# MarkerCluster necesita una lista de (lat, lon) — PPT Slide 8

hogares = list(zip(bono_uni['latitud'], bono_uni['longitud']))
print(f"Puntos para el mapa: {len(hogares)}")
print(f"Ejemplo: {hogares[0]}")


In [ ]:
# ─── PASO 5D: MarkerCluster — agrupar muchos puntos ─────────────────────────
# PPT Slide 8: con miles de puntos, MarkerCluster evita que el mapa se vuelva lento
# Agrupa automáticamente y muestra el conteo por zona

lat_palacio  = -12.0757538
long_palacio = -76.9863174

z = fm.Map(location=[lat_palacio, long_palacio], zoom_start=12)

MarkerCluster(
    hogares,
    name = 'Cluster — Bono Universal'
).add_to(z)

z


In [ ]:
# ─── PASO 5E: HeatMap — mapa de calor por densidad ───────────────────────────
# PPT Slide 8: HeatMap → ideal para ver DÓNDE se concentran los fenómenos
# radius → radio de influencia de cada punto

z = fm.Map(location=[lat_palacio, long_palacio], zoom_start=12)

HeatMap(
    data   = bono_uni[['latitud', 'longitud']],
    radius = 20,                             # radio de influencia (píxeles)
    name   = 'Heatmap — Bono Universal'
).add_to(z)

z


---
## Paso 6 — Markers por NBI: Necesidades Básicas Insatisfechas
>  **PPT Slide 5** — Marcadores + **PPT Slide 8** — Marcadores por categoría  
> **NBIs (ENAHO):** nbi1=Vivienda inadecuada | nbi2=Hacinamiento | nbi3=Sin SSHH | nbi4=Inasistencia escolar | nbi5=Alta dependencia económica


In [ ]:
# ─── PASO 6A: Filtrar hogares con cada NBI ───────────────────────────────────
# Cada NBI es una variable binaria: 1=tiene la carencia, 0=no la tiene

base1 = ENAHO[ENAHO['nbi1'] == 1]    # Vivienda inadecuada
base2 = ENAHO[ENAHO['nbi2'] == 1]    # Hacinamiento
base3 = ENAHO[ENAHO['nbi3'] == 1]    # Sin SSHH
base4 = ENAHO[ENAHO['nbi4'] == 1]    # Inasistencia escolar
base5 = ENAHO[ENAHO['nbi5'] == 1]    # Alta dependencia económica

print("Hogares con cada necesidad básica insatisfecha:")
for i, b in enumerate([base1, base2, base3, base4, base5], 1):
    print(f"  NBI{i}: {len(b):,} hogares")


In [ ]:
# ─── PASO 6B: Circles por NBI — un color por categoría ──────────────────────
# PPT Slide 5: fm.Circle + loop + colors por categoría

m = fm.Map(location=[lat_palacio, long_palacio], zoom_start=12.5, control_scale=True)

colors = ['forestgreen', 'darkred', 'blue', 'lime']

for j in range(1, 5):
    for idx, row in globals()[f'base{j}'].iterrows():
        fm.Circle(
            [row['latitud'], row['longitud']],
            radius = 200,
            color  = colors[j - 1]
        ).add_to(m)

m


In [ ]:
# ─── PASO 6C: Markers con ícono y popup por NBI ──────────────────────────────
# PPT Slide 5 — íconos Font Awesome + popup con el tipo de carencia

m = fm.Map(location=[lat_palacio, long_palacio], zoom_start=11, control_scale=True)

colors = ['purple', 'lightgreen', 'red', 'darkblue', 'orange']
nbi_labels = [
    'Vivienda inadecuada',
    'Vivienda con hacinamiento',
    'Vivienda sin SS.HH',
    'Inasistencia escolar',
    'Alta dependencia económica'
]

for j in range(1, 5):
    for idx, row in globals()[f'base{j}'].iterrows():
        Marker(
            [row['latitud'], row['longitud']],
            icon  = fm.Icon(color=colors[j - 1]),
            popup = nbi_labels[j - 1]         # ← tipo de carencia al hacer click
        ).add_to(m)

m


---
##  Paso 7 — Popups HTML Avanzados: Centros de Salud Solidaridad
>  **PPT Slide 9** — Popups Avanzados: Tablas HTML Embebidas en el Mapa  
> **Fuente:** Programa Solidaridad — SISOL Lima  
> Cada marcador muestra una tabla completa con datos del centro de salud al hacer click.


In [ ]:
# ─── PASO 7A: Cargar datos de Centros Solidaridad ─────────────────────────────

ruta_sol = r"../../../_data/Folium\Solidaridad_Center.csv"

with open(ruta_sol, 'rb') as f:
    det = chardet.detect(f.read())

h_solidaridad = pd.read_csv(ruta_sol, encoding=det['encoding'])

print(f"Centros de salud: {len(h_solidaridad)}")
print(f"Columnas: {list(h_solidaridad.columns)}")
display(h_solidaridad.head(4))


In [ ]:
# ─── PASO 7B: Función que genera HTML por cada centro ─────────────────────────
# PPT Slide 9 — Paso 1: visual_html() crea el contenido HTML de cada popup

def visual_html(i):
    """Genera una tabla HTML con la información del centro de salud i"""
    
    left_col_colour  = "#1C7293"   # color columna izquierda (etiqueta)
    right_col_colour = "#EEF6FA"   # color columna derecha (valor)
    
    html = f"""
    <!DOCTYPE html>
    <html>
    <body>
    <table style="height:auto; width:340px; border-collapse:collapse;">
        <tr>
            <td colspan="2" style="background-color:#04354F; color:white; 
                font-weight:bold; font-size:13px; padding:8px; text-align:center;">
                🏥 {h_solidaridad.iloc[i]['Health_center']}
            </td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Distrito</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['distrito']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Dirección</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['direction']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Horario</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['Schedule']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Teléfono</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['phone']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Especialidades</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['especialidades']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Camas disponibles</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['available_beds']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Prueba COVID</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['Prueba_Covid']}</td>
        </tr>
        <tr>
            <td style="background-color:{left_col_colour}; color:white; padding:5px; font-size:11px;">Centro Vacunación</td>
            <td style="background-color:{right_col_colour}; padding:5px; font-size:11px;">{h_solidaridad.iloc[i]['Centro_vacunacion']}</td>
        </tr>
    </table>
    </body>
    </html>
    """
    return html

# Probar con el primer centro
print("Vista previa del popup HTML:")
print(visual_html(0)[:400], "...")


In [ ]:
# ─── PASO 7C: Centrar el mapa en el promedio de coordenadas ──────────────────
# PPT Slide 9: ubication = promedio de lat/lon de todos los centros

ubication = (
    h_solidaridad['latitud'].mean(),
    h_solidaridad['longitud'].mean()
)
print(f"Centro del mapa: Lat {ubication[0]:.4f} | Lon {ubication[1]:.4f}")


In [ ]:
# ─── PASO 7D: Crear mapa con popups enriquecidos (PPT Slide 9 — Paso 2) ──────
# Anatomía del popup: visual_html(i) → IFrame → Popup → Marker
# branca.element.IFrame embebe el HTML dentro del popup

sol = fm.Map(location=ubication, zoom_start=12)

for i in range(len(h_solidaridad)):
    
    # Paso 1: generar el HTML con los datos de la fila i
    html = visual_html(i)
    
    # Paso 2: crear el IFrame que embebe el HTML (PPT Slide 9)
    iframe = br.element.IFrame(html=html, width=360, height=280)
    
    # Paso 3: convertir IFrame en Popup de Folium
    popup = fm.Popup(iframe, parse_html=True)    # ← parse_html=True es OBLIGATORIO
    
    # Paso 4: agregar Marker con el popup y un ícono de hospital
    fm.Marker(
        [h_solidaridad['latitud'].iloc[i],
         h_solidaridad['longitud'].iloc[i]],
        popup = popup,
        icon  = fm.Icon(color='red', icon='medkit', prefix="fa"),  # ícono médico
        tooltip = h_solidaridad['Health_center'].iloc[i]           # nombre al hover
    ).add_to(sol)


print("   Clic en cada marcador rojo para ver la información del centro")

sol


In [ ]:
# ─── PASO 7E: Variación — HeatMap + Marcadores de hospitales ─────────────────
# Combinar información de densidad con ubicaciones específicas

sol2 = fm.Map(location=ubication, zoom_start=11)

# HeatMap de la distribución de centros
HeatMap(
    data   = h_solidaridad[['latitud', 'longitud']],
    radius = 30,
    name   = 'Densidad de Centros'
).add_to(sol2)

# Markers individuales
for i in range(len(h_solidaridad)):
    html   = visual_html(i)
    iframe = br.element.IFrame(html=html, width=360, height=280)
    popup  = fm.Popup(iframe, parse_html=True)
    
    fm.Marker(
        [h_solidaridad['latitud'].iloc[i], h_solidaridad['longitud'].iloc[i]],
        popup   = popup,
        icon    = fm.Icon(color='blue', icon='medkit', prefix="fa"),
        tooltip = h_solidaridad['distrito'].iloc[i]
    ).add_to(sol2)

fm.LayerControl().add_to(sol2)
sol2.save("centros_solidaridad_heatmap.html")
print("💾 Guardado: centros_solidaridad_heatmap.html")

sol2


---
##  Resumen — Flujo de Trabajo con Folium
>  **PPT Slide 10** — Checklist y errores más comunes

```python
# ─── Los 6 pasos del flujo Folium ─────────────────────────────────────────

# PASO 1 — Encoding
with open(archivo, 'rb') as f:
    charenc = chardet.detect(f.read())['encoding']
df = pd.read_csv(archivo, encoding=charenc)

# PASO 2 — Mapa base
mapa = fm.Map(location=[lat, lon], tiles="cartodbpositron", zoom_start=5)

# PASO 3 — Marcadores (loop iterrows)
for _, row in df.iterrows():
    fm.Marker([row['lat'], row['lon']], popup=row['info']).add_to(mapa)

# PASO 4 — Choropleth
fm.Choropleth(geo_data=geojson, data=df, columns=['ID','VAR'],
              key_on="feature.properties.ID").add_to(mapa)

# PASO 5 — HeatMap / MarkerCluster
HeatMap(data=df[['lat','lon']], radius=20).add_to(mapa)
MarkerCluster(list(zip(df.lat, df.lon))).add_to(mapa)

# PASO 6 — Guardar
fm.LayerControl().add_to(mapa)
mapa.save("mi_mapa.html")
```

| Error | Causa | Solución |
|-------|-------|----------|
| Mapa vacío en choropleth | `key_on` mal escrito | Verificar con `geojson.columns` |
| Caracteres corruptos | Encoding incorrecto | Usar `chardet.detect()` primero |
| Popup no aparece | `parse_html=True` olvidado | `fm.Popup(iframe, parse_html=True)` |
| Mapa lento / bloqueado | Miles de markers sin cluster | Usar `MarkerCluster` |
| Tooltip no visible | Falta `keep_in_front` | `z.keep_in_front(details)` |


---
##  Referencias

**Datos utilizados:**
- MINEM — Centros mineros georreferenciados: http://www.minem.gob.pe/_publicaSector.php?idSector=1&idCategoria=24
- INEI — Mapa de Pobreza distrital: https://www.inei.gob.pe/cifras-de-pobreza/
- GeoPeru — Información geoespacial: https://visor.geoperu.gob.pe/
- INEI DataCrim: https://datacrim.inei.gob.pe/

**Documentación Folium:**
- Folium Docs: https://python-visualization.github.io/folium/index.html
- Kaggle Tutorial: https://www.kaggle.com/alexisbcook/interactive-maps
- Fancy Folium: https://www.kaggle.com/dabaker/fancy-folium
- Choropleth avanzado: https://towardsdatascience.com/how-to-step-up-your-folium-choropleth-map-skills-17cf6de7c6fe

**Geometría del Perú:**
- GeoGPS Perú: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html
- IDE INEI: https://ide.inei.gob.pe/
- HDX: https://data.humdata.org/dataset/cod-ab-per
